# Poglavlje 7: Izgradnja chat aplikacija
## OpenAI API Brzi početak

Ovaj bilježnik je prilagođen iz [Azure OpenAI uzoraka repozitorija](https://github.com/Azure/azure-openai-samples?WT.mc_id=academic-105485-koreyst) koji uključuje bilježnike koji pristupaju [Azure OpenAI](aoai-assignment.ipynb) uslugama.

Python OpenAI API također radi s Azure OpenAI modelima, uz nekoliko izmjena. Saznajte više o razlikama ovdje: [Kako prebaciti između OpenAI i Azure OpenAI krajnjih točaka s Pythonom](https://learn.microsoft.com/azure/ai-foundry/openai/how-to/switching-endpoints?WT.mc_id=academic-109527-jasmineg)


# Pregled  
"Veliki jezični modeli su funkcije koje mapiraju tekst u tekst. Dajući ulazni niz teksta, veliki jezični model pokušava predvidjeti tekst koji će slijediti"(1). Ova "brza početna" bilježnica upoznat će korisnike s visokorazinskim konceptima LLM-a, osnovnim zahtjevima paketa za početak rada s AML-om, laganim uvodom u dizajn upita te nekoliko kratkih primjera različitih slučajeva uporabe. 


## Sadržaj  

[Pregled](#overview)  
[Kako koristiti OpenAI servis](#how-to-use-openai-service)  
[1. Kreiranje vašeg OpenAI servisa](#1.-creating-your-openai-service)  
[2. Instalacija](#2.-installation)    
[3. Podaci za prijavu](#3.-credentials)  

[Primjeri korištenja](#use-cases)    
[1. Sažetak teksta](#1.-summarize-text)  
[2. Klasifikacija teksta](#2.-classify-text)  
[3. Generiranje novih naziva proizvoda](#3.-generate-new-product-names)  
[4. Fino podešavanje klasifikatora](#4.fine-tune-a-classifier)  

[Reference](#references)


### Izgradite svoj prvi prompt  
Ova kratka vježba pružit će osnovni uvod u slanje promptova OpenAI modelu za jednostavan zadatak "sažimanja".


**Koraci**:  
1. Instalirajte OpenAI biblioteku u vašem python okruženju  
2. Učitajte standardne pomoćne biblioteke i postavite vaše uobičajene OpenAI sigurnosne akreditive za OpenAI servis koji ste kreirali  
3. Odaberite model za vaš zadatak  
4. Kreirajte jednostavan prompt za model  
5. Pošaljite vaš zahtjev model API-ju!


### 1. Instalirajte OpenAI


In [ ]:
%pip install openai python-dotenv

### 2. Uvoz pomoćnih biblioteka i instanciranje vjerodajnica


In [ ]:
import os
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()

API_KEY = os.getenv("OPENAI_API_KEY","")
assert API_KEY, "ERROR: OpenAI Key is missing"

client = OpenAI(
    api_key=API_KEY
    )


### 3. Pronalaženje pravog modela  
Modeli poput GPT-4o i GPT-4o mini mogu razumjeti i generirati prirodni jezik, te su dostupni na OpenAI platformi s različitim razinama snage i brzine pogodne za različite zadatke.


In [ ]:
# Select a general purpose chat model
model = "gpt-5-mini"


## 4. Dizajn poticaja  

"Čarolija velikih jezičnih modela je u tome što, trenirajući se da minimiziraju ovu pogrešku predviđanja na golemim količinama teksta, modeli na kraju nauče koncepte korisne za ta predviđanja. Na primjer, oni uče koncepte kao što su"(1):

* kako se piše
* kako gramatika funkcionira
* kako parafrazirati
* kako odgovarati na pitanja
* kako voditi razgovor
* kako pisati na mnogim jezicima
* kako kodirati
* itd.

#### Kako upravljati velikim jezičnim modelom  
"Od svih ulaza u veliki jezični model, daleko najutjecajniji je tekstualni poticaj(1).

Veliki jezični modeli mogu se potaknuti na generiranje izlaza na nekoliko načina:

Instrukcija: Reci modelu što želiš
Dovršetak: Natjeraj model da dovrši početak onoga što želiš
Demonstracija: Pokaži modelu što želiš, s bilo:
Nekoliko primjera u poticaju
Mnoge stotine ili tisuće primjera u skupu podataka za fino podešavanje"



#### Postoje tri osnovne smjernice za stvaranje poticaja:

**Pokaži i reci**. Jasno naznači što želiš bilo putem instrukcija, primjera ili kombinacijom oboje. Ako želiš da model rangira popis stavki abecednim redom ili da klasificira odlomak po sentimentu, pokaži mu da je to što želiš.

**Osiguraj kvalitetne podatke**. Ako pokušavaš napraviti klasifikator ili natjerati model da slijedi neki uzorak, pobrini se da ima dovoljno primjera. Provjeri svoje primjere — model je obično dovoljno pametan da uoči osnovne pravopisne pogreške i odgovori ti, ali također može pretpostaviti da je to namjerno i to može utjecati na odgovor.

**Provjeri svoje postavke.** Postavke temperature i top_p kontroliraju koliko je model determinističan u generiranju odgovora. Ako tražiš odgovor gdje postoji samo jedan točan odgovor, onda želiš postaviti ove vrijednosti niže. Ako tražiš raznovrsnije odgovore, tada ih želiš postaviti više. Najveća pogreška koju ljudi rade s ovim postavkama je pretpostavka da one kontroliraju "pametnost" ili "kreativnost".


Izvor: https://learn.microsoft.com/azure/ai-foundry/openai/overview


### 5. Pošalji!


In [ ]:
# Create your first prompt
text_prompt = "Should oxford commas always be used?"

response = client.responses.create(
  model=model,
  input = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":text_prompt},],
  store=False,)

response.output_text


### Ponovite isti poziv, kako se rezultati uspoređuju?


In [ ]:

response = client.responses.create(
  model=model,
  input = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":text_prompt},],
  store=False,)

response.output_text


## Sažetak teksta  
#### Izazov  
Sažeti tekst dodavanjem 'tl;dr:' na kraj odlomka teksta. Primjetite kako model razumije kako izvršiti niz zadataka bez dodatnih uputa. Možete eksperimentirati s opisnijim upitima od tl;dr kako biste izmijenili ponašanje modela i prilagodili sažetak koji dobivate(3).  

Nedavni radovi pokazali su značajan napredak na mnogim NLP zadacima i standardima pomoću prethodnog učenja na velikom korpusu teksta, praćenog finim podešavanjem za specifični zadatak. Iako je arhitektura obično nezavisna od zadatka, ova metoda i dalje zahtijeva skupove podataka za fino podešavanje specifične za zadatak od tisuća ili desetaka tisuća primjera. Suprotno tome, ljudi općenito mogu izvršiti novi jezični zadatak iz samo nekoliko primjera ili jednostavnih uputa - što trenutačni NLP sustavi uglavnom još uvijek teško mogu. Ovdje pokazujemo da povećanje jezičnih modela znatno poboljšava izvedbu bez zadatka, s malo primjera, ponekad dosežući konkurentnost s dotadašnjim vrhunskim pristupima finom podešavanju.  



Tl;dr  


# Vježbe za nekoliko slučajeva upotrebe  
1. Sažetak teksta  
2. Klasificirajte tekst  
3. Generiranje novih imena proizvoda


In [ ]:
prompt = "Recent work has demonstrated substantial gains on many NLP tasks and benchmarks by pre-training on a large corpus of text followed by fine-tuning on a specific task. While typically task-agnostic in architecture, this method still requires task-specific fine-tuning datasets of thousands or tens of thousands of examples. By contrast, humans can generally perform a new language task from only a few examples or from simple instructions - something that current NLP systems still largely struggle to do. Here we show that scaling up language models greatly improves task-agnostic, few-shot performance, sometimes even reaching competitiveness with prior state-of-the-art fine-tuning approaches.\n\nTl;dr"


In [ ]:
#Setting a few additional, typical parameters during API Call

response = client.responses.create(
  model=model,
  input = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":prompt},],
  store=False,)

response.output_text


## Klasificiraj tekst  
#### Izazov  
Klasificiraj stavke u kategorije navedene u trenutku izvođenja. U sljedećem primjeru dajemo i kategorije i tekst za klasifikaciju u promptu (*playground_reference). 

Upit korisnika: Pozdrav, jedna od tipki na tipkovnici mog prijenosnog računala nedavno se pokvarila i trebat će mi zamjena:

Klasificirana kategorija:


In [ ]:
prompt = "Classify the following inquiry into one of the following: categories: [Pricing, Hardware Support, Software Support]\n\ninquiry: Hello, one of the keys on my laptop keyboard broke recently and I'll need a replacement:\n\nClassified category:"
print(prompt)

In [ ]:
#Setting a few additional, typical parameters during API Call

response = client.responses.create(
  model=model,
  input = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":prompt},],
  store=False,)

response.output_text


## Generirajte Nove Nazive Proizvoda
#### Izazov
Izradite nazive proizvoda na temelju primjera riječi. Ovdje u upit uključujemo informacije o proizvodu za koji ćemo generirati nazive. Također dajemo sličan primjer kako bismo pokazali obrazac koji želimo dobiti. Također smo postavili visoku vrijednost temperature kako bismo povećali nasumičnost i dobili inovativnije odgovore.

Opis proizvoda: Kućni aparat za milkshake
Početne riječi: brz, zdrav, kompaktan.
Nazivi proizvoda: HomeShaker, Fit Shaker, QuickShake, Shake Maker

Opis proizvoda: Par cipela koji pristaje na bilo koju veličinu stopala.
Početne riječi: prilagodljiv, odgovara, omni-fit.


In [ ]:
prompt = "Product description: A home milkshake maker\nSeed words: fast, healthy, compact.\nProduct names: HomeShaker, Fit Shaker, QuickShake, Shake Maker\n\nProduct description: A pair of shoes that can fit any foot size.\nSeed words: adaptable, fit, omni-fit."

print(prompt)

In [ ]:
#Setting a few additional, typical parameters during API Call

response = client.responses.create(
  model=model,
  input = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":prompt}],
  store=False,)

response.output_text


# Reference  
- [Openai Cookbook](https://github.com/openai/openai-cookbook?WT.mc_id=academic-105485-koreyst)  
- [Microsoft Foundry portal](https://ai.azure.com?WT.mc_id=academic-105485-koreyst)  
- [Najbolje prakse za fino podešavanje GPT modela za klasifikaciju teksta](https://platform.openai.com/docs/guides/fine-tuning?WT.mc_id=academic-105485-koreyst)


# Za Više Pomoći  
[OpenAI Tim za Komercijalizaciju](AzureOpenAITeam@microsoft.com) 


# Suradnici
* Louis Li  


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Napomena**:
Ovaj dokument je preveden korištenjem AI prevoditeljskog servisa [Co-op Translator](https://github.com/Azure/co-op-translator). Iako težimo točnosti, imajte na umu da automatski prijevodi mogu sadržavati greške ili netočnosti. Izvorni dokument na izvornom jeziku treba smatrati autoritativnim izvorom. Za važne informacije preporuča se profesionalni ljudski prijevod. Nismo odgovorni za bilo kakva nesporazumevanja ili pogrešne interpretacije koje proizlaze iz korištenja ovog prijevoda.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
